In [5]:
import pandas as pd
import sqlite3

# EXECUTE SETUP TO INSTANTIATE DATA CONTEXTS
conn = sqlite3.connect(":memory:")
servers_inventory = {
    "Host_ID": ["SRV-01", "SRV-02", "SRV-03"],
    "Role": ["Web Front", "API Gateway", "Database Replica"]
}
live_interfaces = {
    "Interface_ID": ["eth0", "eth1"],
    "Mapped_Host": ["SRV-01", "SRV-02"],
    "IP_Address": ["10.0.0.4", "10.0.0.9"]
}
 
df_srv = pd.DataFrame(servers_inventory)
df_inf = pd.DataFrame(live_interfaces)
 
df_srv.to_sql('Servers', conn, index=False, if_exists='replace')
df_inf.to_sql('Interfaces', conn, index=False, if_exists='replace')
 
# Broken Query Framework
faulty_query = """
SELECT s.Host_ID, s.Role, i.Interface_ID, i.IP_Address
FROM Servers s
INNER JOIN Interfaces i ON s.Host_ID = i.Mapped_Host;
"""
print("--- Faulty Output Metrics (Missing Database Replica SRV-03) ---")
pd.read_sql_query(faulty_query, conn)



--- Faulty Output Metrics (Missing Database Replica SRV-03) ---


,Host_ID,Role,Interface_ID,IP_Address
0,SRV-01,Web Front,eth0,10.0.0.4
1,SRV-02,API Gateway,eth1,10.0.0.9


In [11]:
# Correct Query (Include ALL Servers)
# To include all servers, even those without interfaces, use a LEFT JOIN.

query = """
SELECT s.Host_ID, s.Role, i.Interface_ID, i.IP_Address
FROM Servers s
LEFT JOIN Interfaces i
ON s.Host_ID = i.Mapped_Host;
"""
print(pd.read_sql_query (query, conn))

  Host_ID              Role Interface_ID IP_Address
0  SRV-01         Web Front         eth0   10.0.0.4
1  SRV-02       API Gateway         eth1   10.0.0.9
2  SRV-03  Database Replica         None       None


In [ ]:

# Explanation:
# The INNER JOIN returns only rows where there is a match between
# Servers.Host_ID and Interfaces.Mapped_Host.
#
# Since SRV-03 exists in the Servers table but does NOT have a matching
# entry in the Interfaces table, it gets excluded from the result.
#
# To include all servers (even those without interfaces like SRV-03),
# a LEFT JOIN should be used instead of INNER JOIN.
# A LEFT JOIN returns all records from the left table (Servers)
# and fills NULL for missing matches from the right table (Interfaces).



